[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.1 MB/s eta 0:00:00


In [2]:
import torch

In [12]:
# core code
def batch_select(tensor, indices):
    """Select specific indices from each batch element using gather"""
    # tensor: [batch_size, sequence_length, features]
    # indices: [batch_size, num_selections]
    batch_size, sequence_length, features = tensor.shape
    _, num_selections = indices.shape
    indices = indices.unsqueeze(-1).expand(batch_size, num_selections, features)
    selected = torch.gather(tensor, 1, indices)
    return selected

# Example: Select specific timesteps from sequence data
batch_size, seq_len, features = 3, 10, 4
sequences = torch.randn(batch_size, seq_len, features)
indices = torch.tensor([[0, 5, 9],   # Select timesteps 0, 5, 9 from batch 0
                        [1, 3, 7],   # Select timesteps 1, 3, 7 from batch 1
                        [2, 4, 8]])  # Select timesteps 2, 4, 8 from batch 2

selected = batch_select(sequences, indices)
print(f"Input shape: {sequences.shape}")
print(f"Indices shape: {indices.shape}")
print(f"Selected shape: {selected.shape}")

# Verify selection
for b in range(batch_size):
    for i, idx in enumerate(indices[b]):
        assert torch.allclose(selected[b, i], sequences[b, idx]), f"Failed at batch {b}, selection {i}"

print("\n✓ Batch selection verified")


Input shape: torch.Size([3, 10, 4])
Indices shape: torch.Size([3, 3])
Selected shape: torch.Size([3, 3, 4])

✓ Batch selection verified


In [13]:
x = torch.tensor([[1], [2], [3]])
print(x.shape)

torch.Size([3, 1])


In [140]:
# ✏️ YOUR IMPLEMENTATION HERE

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    log_probs = torch.zeros(beam_width,)
    beams = torch.tensor(start_token).unsqueeze(0).unsqueeze(1)
    for s in range(max_len):
        if s > 0:
          rows = [log_prob_fn(beams[b]) for b in range(beam_width)]
        else:
          rows = [log_prob_fn(beams[b]) for b in range(1)]
        new_log_probs = torch.stack(rows, dim=0)
        new_log_probs = new_log_probs + log_probs.unsqueeze(1)
        values, indices = torch.topk(new_log_probs.flatten(), beam_width, dim=-1)
        rindices = indices // new_log_probs.shape[1] # what beam was selected
        cindices = indices % new_log_probs.shape[1] # what word was selected
        log_probs = values
        print(beams.shape, rindices.unsqueeze(1))
        print(rindices.unsqueeze(1).expand(beam_width, beams.shape[1]).shape)
        print(torch.gather(beams, 0, rindices.unsqueeze(1).expand(beam_width, beams.shape[1])))
        beams = torch.cat((torch.gather(beams, 0, rindices.unsqueeze(1).expand(beam_width, beams.shape[1])), cindices.unsqueeze(1)), -1)
        print(beams)
    beam = beams[torch.max(log_probs, dim=0).indices].tolist()
    return beams.tolist()
    pass  # maintain beams, expand, prune, return best

In [172]:
# ✏️ YOUR IMPLEMENTATION HERE

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    log_probs = torch.zeros(1)
    beams = torch.tensor(start_token).unsqueeze(0).unsqueeze(1)
    for s in range(max_len):
        rows = [log_prob_fn(beams[b]) for b in range(beams.shape[0])]
        new_log_probs = torch.stack(rows, dim=0)
        new_log_probs = new_log_probs + log_probs.unsqueeze(1)
        values, indices = torch.topk(new_log_probs.flatten(), beam_width, dim=-1)
        rindices = indices // new_log_probs.shape[1] # what beam was selected
        cindices = indices % new_log_probs.shape[1] # what word was selected
        log_probs = values
        beams = torch.cat((torch.gather(beams, 0, rindices.unsqueeze(1).expand(beam_width, beams.shape[1])), cindices.unsqueeze(1)), -1)
        # print(beams)
    beam = beams[torch.max(log_probs, dim=0).indices].tolist()
    return beam
    # pass  # maintain beams, expand, prune, return best

    #     print(new_log_probs.shape, values.shape, rindices.shape, )
    #     print(beams.shape, rindices.unsqueeze(1).expand(beam_width, beams.shape[1]).shape)
    #     print(rindices.unsqueeze(1).expand(beam_width, beams.shape[1]))
    #     return


In [173]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

Sequence: [0, 1, 2, 3, 4, 4]


In [174]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')


🧪 Testing: Beam Search Decoding (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Returns list starting with start_token (3.5ms)
  ❌ [2/4] Greedy path (beam=1)
     Greedy: [0, 1, 2, 3, 4, 4]
  ❌ [3/4] Beam finds better path than greedy
     Beam should find [0,1,5], got [0, 1, 5, 5, 5, 5]
  ❌ [4/4] Stops at eos
     Should be [0,3], got [0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
──────────────────────────────────────────────────
  📊 1/4 tests passed.
  Keep going! Use hint("beam_search") if you're stuck.

